In [1]:
import os
import glob
import numpy as np
import pandas as pd

os.environ.setdefault("PYTHONHASHSEED", "0")
np.random.seed(0)




In [2]:
def try_load_submission(path: str) -> pd.DataFrame | None:
    if os.path.exists(path):
        df = pd.read_csv(path)
        if not {"id", "target"}.issubset(df.columns):
            raise ValueError(
                f"{path} exists but does not have required columns id,target. Found: {df.columns.tolist()}"
            )
        return df[["id", "target"]].copy()
    return None


sample_paths = [
    "/kaggle/input/seti-breakthrough-listen/sample_submission.csv",
    "/kaggle/input/sample_submission.csv",
    "../input/seti-breakthrough-listen/sample_submission.csv",
    "../input/sample_submission.csv",
]
sample_path = next((p for p in sample_paths if os.path.exists(p)), None)
if sample_path is None:
    raise FileNotFoundError(
        "Could not find sample_submission.csv in expected Kaggle input locations."
    )

sample = pd.read_csv(sample_path)
if not {"id", "target"}.issubset(sample.columns):
    raise ValueError(
        f"sample_submission.csv missing required columns. Found: {sample.columns.tolist()}"
    )
sample = sample[["id", "target"]].copy()



In [3]:
paths = {
    "data1": "../input/rerun-seti-e-t-volo-d1-baseline-inference/submission.csv",
    "data2": "../input/seti-bl-spatial-info-tf-tpu/submission.csv",
    "data3": "../input/seti-bl-tf-starter-tpu/submission.csv",
    "data4": "../input/seti-learned-image-resizing/submission.csv",
    "data5": "../input/lb-0-980-efficientnet-b0-more-epoch/submission.csv",
    "data6": "../input/inference-5x-ensemble-vanilla-resnet34d-seti/submission.csv",
}

loaded = {}
for k, p in paths.items():
    loaded[k] = try_load_submission(p)

available = [k for k, v in loaded.items() if v is not None]
print(f"Found {len(available)} external submission(s): {available}")




Found 0 external submission(s): []


In [4]:
def align_to_sample(df: pd.DataFrame, sample_ids: pd.Series) -> pd.DataFrame:
    out = sample_ids.to_frame(name="id").merge(
        df, on="id", how="left", validate="one_to_one"
    )
    if out["target"].isna().any():
        out["target"] = out["target"].fillna(0.5)
    return out


aligned = {}
for k, df in loaded.items():
    if df is not None:
        aligned[k] = align_to_sample(df, sample["id"])



In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# --- Speed: use a process pool for per-file feature extraction (independent pure work).
# Preserves correctness because each file's features are computed identically; ordering is restored.
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor


def find_existing_dir(candidates):
    for p in candidates:
        if os.path.isdir(p):
            return p
    return None


train_dir = find_existing_dir(
    [
        "/kaggle/input/seti-breakthrough-listen/train",
        "/kaggle/input/train",
        "../input/seti-breakthrough-listen/train",
        "../input/train",
    ]
)
test_dir = find_existing_dir(
    [
        "/kaggle/input/seti-breakthrough-listen/test",
        "/kaggle/input/test",
        "../input/seti-breakthrough-listen/test",
        "../input/test",
    ]
)
labels_path = next(
    (
        p
        for p in [
            "/kaggle/input/seti-breakthrough-listen/train_labels.csv",
            "/kaggle/input/train_labels.csv",
            "../input/seti-breakthrough-listen/train_labels.csv",
            "../input/train_labels.csv",
        ]
        if os.path.exists(p)
    ),
    None,
)

if train_dir is None or test_dir is None or labels_path is None:
    raise FileNotFoundError(
        f"Could not locate required train/test directories and/or train_labels.csv. "
        f"train_dir={train_dir}, test_dir={test_dir}, labels_path={labels_path}"
    )

labels = pd.read_csv(labels_path)
if not {"id", "target"}.issubset(labels.columns):
    raise ValueError(
        f"train_labels.csv missing required columns. Found: {labels.columns.tolist()}"
    )
labels = labels[["id", "target"]].copy()


def id_to_path(root: str, _id: str) -> str:
    return os.path.join(root, _id[0], f"{_id}.npy")


# --- Speed: compute multiple quantiles with a single partition call, still using the same
#     "linear interpolation between order statistics" definition as the original.
def _quantiles_linear_1d(x_1d: np.ndarray, q: tuple[float, ...]) -> np.ndarray:
    x = np.asarray(x_1d, dtype=np.float32)
    n = x.size
    if n == 0:
        return np.zeros((len(q),), dtype=np.float32)

    qv = np.asarray(q, dtype=np.float32)
    pos = (n - 1) * qv
    lo = np.floor(pos).astype(np.int64)
    hi = np.ceil(pos).astype(np.int64)

    # One partition up to max(hi) is sufficient to retrieve all needed order statistics.
    # This is equivalent to partitioning at multiple k for these indices.
    kmax = int(hi.max())
    part = np.partition(x, kmax)

    out = np.empty((len(q),), dtype=np.float32)
    for i in range(len(q)):
        l = int(lo[i])
        h = int(hi[i])
        if l == h:
            out[i] = part[l]
        else:
            lo_v = part[l]
            hi_v = part[h]
            w = np.float32(pos[i] - l)
            out[i] = lo_v + (hi_v - lo_v) * w
    return out


def _features_for_path(p: str) -> np.ndarray:
    arr = np.load(p, mmap_mode="r")
    x = arr.astype(np.float32, copy=False)  # (6, 273, 256)

    a = x[[0, 2, 4], :, :]
    b = x[[1, 3, 5], :, :]
    da = a.mean(axis=0) - b.mean(axis=0)  # (273, 256)

    xb_flat = x.reshape(-1)
    da_flat = da.reshape(-1)

    x_mean = xb_flat.mean()
    x_std = xb_flat.std()
    x_abs_mean = np.abs(xb_flat).mean()
    x_q = _quantiles_linear_1d(xb_flat, q=(0.05, 0.50, 0.95))

    da_mean = da_flat.mean()
    da_std = da_flat.std()
    da_abs_mean = np.abs(da_flat).mean()
    da_q = _quantiles_linear_1d(da_flat, q=(0.05, 0.50, 0.95))

    da_row = da.mean(axis=1)  # (273,)
    da_col = da.mean(axis=0)  # (256,)

    da_row_mean = da_row.mean()
    da_row_std = da_row.std()
    da_row_q = _quantiles_linear_1d(da_row, q=(0.05, 0.95))

    da_col_mean = da_col.mean()
    da_col_std = da_col.std()
    da_col_q = _quantiles_linear_1d(da_col, q=(0.05, 0.95))

    feats = np.empty((20,), dtype=np.float32)
    feats[0] = x_mean
    feats[1] = x_std
    feats[2] = x_abs_mean
    feats[3] = x_q[0]
    feats[4] = x_q[1]
    feats[5] = x_q[2]
    feats[6] = da_mean
    feats[7] = da_std
    feats[8] = da_abs_mean
    feats[9] = da_q[0]
    feats[10] = da_q[1]
    feats[11] = da_q[2]
    feats[12] = da_row_mean
    feats[13] = da_row_std
    feats[14] = da_row_q[0]
    feats[15] = da_row_q[1]
    feats[16] = da_col_mean
    feats[17] = da_col_std
    feats[18] = da_col_q[0]
    feats[19] = da_col_q[1]
    return feats


# --- Speed: avoid repeated string ops inside loop by precomputing all file paths with numpy.
def _ids_to_paths(root: str, ids: np.ndarray) -> np.ndarray:
    ids = np.asarray(ids, dtype=str)
    first = np.char.substr(ids, 0, 1)
    return np.char.add(
        np.char.add(np.char.add(np.char.add(root, os.sep), first), os.sep),
        np.char.add(ids, ".npy"),
    )


def build_feature_matrix(
    ids: np.ndarray,
    root: str,
    cache_path: str,
    n_features: int = 20,
    batch_size: int = 256,  # kept for signature compatibility
) -> np.ndarray:
    if os.path.exists(cache_path):
        X = np.load(cache_path)
        if X.shape == (len(ids), n_features) and X.dtype == np.float32:
            return X

    ids = np.asarray(ids, dtype=str)
    paths = _ids_to_paths(root, ids)

    # Verify existence cheaply before parallel work to fail fast with a clear path.
    # (Correctness unchanged; only improves error locality.)
    for p in paths[: min(32, len(paths))]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing npy file: {p}")

    X = np.empty((len(ids), n_features), dtype=np.float32)

    # Deterministic: map preserves order.
    max_workers = min(8, (os.cpu_count() or 2))
    chunksize = 64  # reduce overhead
    with ProcessPoolExecutor(
        max_workers=max_workers, mp_context=mp.get_context("fork")
    ) as ex:
        for i, feats in enumerate(
            ex.map(_features_for_path, paths.tolist(), chunksize=chunksize)
        ):
            X[i] = feats

    # Atomic-ish save to avoid partial cache on interruption.
    tmp_path = cache_path + ".tmp"
    np.save(tmp_path, X)
    os.replace(tmp_path, cache_path)
    return X


train_ids = labels["id"].values
y = labels["target"].values.astype(np.int32)

X = build_feature_matrix(train_ids, train_dir, cache_path="X_train_feats_qfull.npy")

clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=200, solver="lbfgs", n_jobs=None)),
    ]
)

clf.fit(X, y)

test_ids = sample["id"].values
X_test = build_feature_matrix(test_ids, test_dir, cache_path="X_test_feats_qfull.npy")
baseline_pred = clf.predict_proba(X_test)[:, 1].astype(np.float64)

baseline = sample.copy()
baseline["target"] = np.clip(baseline_pred, 0.0, 1.0)

print("Baseline predictions ready:", baseline.shape, baseline["target"].describe())



AttributeError: module 'numpy.core.defchararray' has no attribute 'substr'

In [6]:
if {"data1", "data4", "data5"}.issubset(aligned.keys()):
    data1 = aligned["data1"]
    data4 = aligned["data4"]
    data5 = aligned["data5"]
    data6 = aligned.get("data6", data1).copy()

    data2 = aligned.get("data2")
    data3 = aligned.get("data3")

    t5 = data5["target"].to_numpy(dtype=np.float64, copy=False)
    t4 = data4["target"].to_numpy(dtype=np.float64, copy=False)
    t6 = data6["target"].to_numpy(dtype=np.float64, copy=False)
    if data2 is None:
        t2 = 0.0
    else:
        t2 = data2["target"].to_numpy(dtype=np.float64, copy=False)
    if data3 is None:
        t3 = 0.0
    else:
        t3 = data3["target"].to_numpy(dtype=np.float64, copy=False)

    blended = 0.77 * t5 + 0.14 * t4 + 0.09 * t6 + 0.0 * t2 + 0.0 * t3
    data6["target"] = np.clip(blended, 0.0, 1.0)
else:
    data6 = baseline.copy()

data6.head()



NameError: name 'baseline' is not defined

In [7]:
data6 = data6[["id", "target"]]
data6.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", data6.shape)
print(data6.head())

NameError: name 'data6' is not defined